# Graph-Attentive Neural Hawkes for Wallet Classification

A hybrid architecture that fuses three established mechanisms into one per-event update:
1. **Continuous-time LSTM** (Mei & Eisner 2017 *Neural Hawkes*) — provides time-decay between events.
2. **Masked temporal self-attention with time-decay bias** — looks back over a node's own past hidden states.
3. **Heterogeneous graph attention** over current neighbours — uses the actor-graph's three relations (`AddrAddr`, `AddrTx`, `TxAddr`) plus the tx ↔ tx edge list from `transactions_data/`.

## Per-event update for wallet $i$ at time $t_k$

**Decay** the cell state from the last event:
$$\mathbf{c}_i(t_k^-) = \bar{\mathbf{c}}_i + (\mathbf{c}_i(t_{k-1}) - \bar{\mathbf{c}}_i) \cdot e^{-\boldsymbol{\delta}_i (t_k - t_{k-1})}, \quad \mathbf{h}_i(t_k^-) = \mathbf{o}_i \odot \tanh \mathbf{c}_i(t_k^-)$$

**Temporal self-attention** over the wallet's past hidden states with additive time-decay bias:
$$\mathbf{z}^{\text{time}}_i = \mathrm{softmax}_j\!\left(\frac{\mathbf{q}_i^\top \mathbf{k}_{i,j}}{\sqrt{d}} - \mathrm{softplus}(\beta_h)\,(t_k - t_j)\right) \mathbf{V}_{i,:}$$

**Heterogeneous graph attention** over current neighbours via {`AddrAddr`, `AddrTx`, `TxAddr`, `tx_to_tx`}:
$$\mathbf{z}^{\text{graph}}_i = \mathrm{HeteroAttn}\!\left(\mathbf{h}_i(t_k^-), \mathcal{N}(i, t_k)\right)$$

**Fuse and update** the CT-LSTM:
$$\mathbf{x}_i(t_k) = \mathbf{W}_{\text{fuse}}[\mathbf{z}^{\text{time}}_i \| \mathbf{z}^{\text{graph}}_i], \quad (\mathbf{c}_i, \bar{\mathbf{c}}_i, \boldsymbol{\delta}_i, \mathbf{o}_i, \mathbf{h}_i)(t_k) = \mathrm{CTLSTM}(\mathbf{x}_i(t_k), \mathbf{h}_i(t_k^-))$$

## Why both attention streams
- **Temporal alone** remembers "I was suspicious at $t=3$" but doesn't know *why*.
- **Graph alone** sees current neighbours but has no memory.
- **Together**: the temporal stream remembers a past suspicion; the graph stream sees the once-suspicious neighbour returning; the fused CT-LSTM input combines both.

## Engineering choices vs the naive description

| Naive | This notebook | Why |
|---|---|---|
| 49 fresh `HeteroData` snapshots | One static graph + per-step edge **active masks** | Edge files have no `Time step`; rebuilding 49× would be wasted O(\|E\|) work. |
| Same machinery on txs | Wallets get full CT-LSTM dynamics; txs get a single graph-attention pass at birth | Txs appear at exactly one time step — no event sequence. |
| `dict[idx] -> list[Tensor]` cache | Padded ring-buffer `[N_subset, W, H]` tensor | Vectorised gather is ~50× faster than Python dict lookups. |
| Pure GAT on every relation | `GATConv` for same-type, `SAGEConv` for cross-type | ~3× faster; cross-type attention rarely earns its compute cost. |
| Full 49-step autograd graph | Truncated BPTT every K=5 steps | Avoids OOM on backward over a 49-step CT-LSTM + attention stack. |

Train/val/test split: time steps **1–29 / 30–34 / 35–49** (chronological). Loss = weighted cross-entropy on the illicit class, applied at **every** wallet event (not terminal-only) — doubles positive supervision.

In [10]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, f1_score
from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, GATConv, SAGEConv
from torch_geometric.transforms import ToUndirected

warnings.filterwarnings("ignore")
torch.manual_seed(42)
np.random.seed(42)

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "transactions_data").exists():
    ROOT = ROOT.parent
TRANSACTIONS_DIR = ROOT / "transactions_data"
WALLETS_DIR = ROOT / "actors_data"

TRAIN_END = 29
VAL_END = 34
N_TIME_STEPS = 49
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"root={ROOT}  device={DEVICE}")

root=/Users/jarayliu/Documents/GitHub/stat175-final  device=cpu


## 1. Load data, build static `HeteroData`, precompute per-step active masks (single cell)

We build:
- `transactions` and `wallet_classes` DataFrames; `wallet_per_step_raw` is the per-`(address, Time step)` feature table.
- A static `HeteroData` with two node types and four directed relations (`AddrAddr`, `AddrTx`, `TxAddr`, `tx_to_tx`); `ToUndirected()` adds reverse relations.
- `active_wallet[T+1, N_w]` and `active_tx[T+1, N_tx]` boolean tensors (1-indexed time steps).
- `edge_active_mask[rel][T+1, |E_rel|]` boolean tensor per relation: an edge is *active at t* iff both endpoints are active at `t`.
- **`step_wallets[t]`** (long tensor of wallet indices active at `t`) and **`step_features[t]`** (their standardised feature rows at time `t`). These drive a **rolling causal feature buffer** in the training loop — at each step `t` the buffer is updated with active wallets' time-`t` features. Inactive wallets keep their most recent snapshot at time `≤ t` (or zeros if never seen). This is what removes the latest-time-step leakage: the GNN at step `t` only ever sees feature rows from time `≤ t`.
- `active_subidx_per_step[t]`: subset wallet indices to update at step `t`.

**Subset:** all illicit wallets (~14K) + 40K random licit wallets. The full graph is preserved for context; the subset only governs which wallets get per-event CT-LSTM state.

**Standardisation:** the wallet scaler is fit on per-step rows with `Time step ≤ TRAIN_END`. The tx scaler is fit on tx rows with `Time step ≤ TRAIN_END`. Neither sees test-window stats.

In [11]:
def map_edges(edge_df, src_col, dst_col, src_map, dst_map):
    src = edge_df[src_col].map(src_map)
    dst = edge_df[dst_col].map(dst_map)
    valid = src.notna() & dst.notna()
    return torch.tensor(
        np.stack([src[valid].astype(np.int64).values, dst[valid].astype(np.int64).values]),
        dtype=torch.long,
    )

# === Transactions ===
transactions = pd.read_csv(TRANSACTIONS_DIR / "txs_features.csv")
tx_classes = pd.read_csv(TRANSACTIONS_DIR / "txs_classes.csv")
tx_classes["label"] = tx_classes["class"].map({1: 1, 2: 0, 3: -1}).astype(np.int8)
transactions = transactions.merge(tx_classes[["txId", "label"]], on="txId", how="left")
transactions["label"] = transactions["label"].fillna(-1).astype(np.int8)
tx_feat_cols = [c for c in transactions.columns if c not in ("txId", "Time step", "label")]
transactions[tx_feat_cols] = transactions[tx_feat_cols].fillna(0).astype(np.float32)
tx_id_to_idx = {tid: i for i, tid in enumerate(transactions["txId"].values)}
tx_features_array = transactions[tx_feat_cols].values
tx_time_steps_np = transactions["Time step"].values.astype(np.int64)

# Tx scaler fit on training-window rows only (each tx has one Time step → no leakage)
tx_train_mask = tx_time_steps_np <= TRAIN_END
tx_features_std = StandardScaler().fit(tx_features_array[tx_train_mask]).transform(tx_features_array).astype(np.float32)

# === Wallet labels (per address; immutable) ===
wallet_classes = pd.read_csv(WALLETS_DIR / "wallets_classes.txt")
wallet_classes["label"] = wallet_classes["class"].map({1: 1, 2: 0, 3: -1}).astype(np.int8)
wallet_addr_to_idx = {addr: i for i, addr in enumerate(wallet_classes["address"].values)}
N_w = len(wallet_classes)
wallet_labels_np = wallet_classes["label"].values
wallet_is_labelled = wallet_labels_np != -1

# === Wallet per-step features ===
with open(WALLETS_DIR / "wallets_features.txt") as f:
    wallet_columns = f.readline().rstrip("\n").split(",")
wallet_dtype = {c: np.float32 for c in wallet_columns if c not in ("address", "Time step")}
wallet_dtype["address"] = "string"
wallet_dtype["Time step"] = np.int16
wallet_per_step_raw = pd.read_csv(WALLETS_DIR / "wallets_features.txt", dtype=wallet_dtype)
wallet_feat_cols = [c for c in wallet_per_step_raw.columns if c not in ("address", "Time step")]
wallet_per_step_raw[wallet_feat_cols] = wallet_per_step_raw[wallet_feat_cols].fillna(0).astype(np.float32)

wp_addr_idx = wallet_per_step_raw["address"].map(wallet_addr_to_idx).astype(np.int64).values
wp_t        = wallet_per_step_raw["Time step"].values.astype(np.int64)
wp_features = wallet_per_step_raw[wallet_feat_cols].values.astype(np.float32)

# Wallet scaler fit on per-step training-window rows only — no leakage from test window
wp_train_mask = wp_t <= TRAIN_END
wp_features_std = StandardScaler().fit(wp_features[wp_train_mask]).transform(wp_features).astype(np.float32)

N_tx = len(transactions)
F_wallet = wp_features.shape[1]

# === Per-step active masks ===
active_wallet = torch.zeros(N_TIME_STEPS + 1, N_w, dtype=torch.bool)
active_wallet[wp_t, wp_addr_idx] = True

active_tx = torch.zeros(N_TIME_STEPS + 1, N_tx, dtype=torch.bool)
active_tx[tx_time_steps_np, np.arange(N_tx)] = True

# === Per-step (wallet_idx, features) lookup for the causal buffer ===
step_wallets = [torch.empty(0, dtype=torch.long) for _ in range(N_TIME_STEPS + 1)]
step_features = [torch.empty(0, F_wallet, dtype=torch.float32) for _ in range(N_TIME_STEPS + 1)]
for t in range(1, N_TIME_STEPS + 1):
    mask = wp_t == t
    if mask.any():
        step_wallets[t]  = torch.tensor(wp_addr_idx[mask], dtype=torch.long)
        step_features[t] = torch.tensor(wp_features_std[mask])

del wallet_per_step_raw, wp_features, wp_features_std

# === Edge files ===
addr_addr_edges = pd.read_csv(WALLETS_DIR / "AddrAddr_edgelist.txt")
addr_tx_edges   = pd.read_csv(WALLETS_DIR / "AddrTx_edgelist.txt")
tx_addr_edges   = pd.read_csv(WALLETS_DIR / "TxAddr_edgelist.txt")
tx_tx_edges     = pd.read_csv(TRANSACTIONS_DIR / "txs_edgelist.csv")

# === Static HeteroData. Wallet `x` is a zeros placeholder (the real per-step features
#     come from the rolling causal buffer in the training/profiler cells). ===
hetero_graph = HeteroData()
hetero_graph["wallet"].x = torch.zeros(N_w, F_wallet, dtype=torch.float32)
hetero_graph["tx"].x     = torch.from_numpy(tx_features_std)
hetero_graph["wallet", "addr_to_addr", "wallet"].edge_index = map_edges(addr_addr_edges, "input_address", "output_address", wallet_addr_to_idx, wallet_addr_to_idx)
hetero_graph["wallet", "addr_to_tx",   "tx"].edge_index     = map_edges(addr_tx_edges,   "input_address", "txId",            wallet_addr_to_idx, tx_id_to_idx)
hetero_graph["tx",     "tx_to_addr",   "wallet"].edge_index = map_edges(tx_addr_edges,   "txId",          "output_address",  tx_id_to_idx,       wallet_addr_to_idx)
hetero_graph["tx",     "tx_to_tx",     "tx"].edge_index     = map_edges(tx_tx_edges,     "txId1",         "txId2",           tx_id_to_idx,       tx_id_to_idx)
hetero_graph = ToUndirected()(hetero_graph)

# === Per-relation per-step edge active masks ===
active_by_type = {"wallet": active_wallet, "tx": active_tx}
edge_active_mask = {}
for etype in hetero_graph.edge_types:
    s_type, _, d_type = etype
    src_active = active_by_type[s_type]
    dst_active = active_by_type[d_type]
    src, dst = hetero_graph[etype].edge_index[0], hetero_graph[etype].edge_index[1]
    edge_active_mask[etype] = src_active[:, src] & dst_active[:, dst]

# === Subset selection (all illicit + 40K licit) ===
labelled_global = np.where(wallet_is_labelled)[0]
illicit_global  = labelled_global[wallet_labels_np[labelled_global] == 1]
licit_global    = labelled_global[wallet_labels_np[labelled_global] == 0]
rng = np.random.default_rng(42)
n_licit_keep = min(40_000, len(licit_global))
subset_global = np.concatenate([illicit_global, rng.choice(licit_global, size=n_licit_keep, replace=False)])
subset_global.sort()
N_subset = len(subset_global)

subset_to_global = torch.tensor(subset_global, dtype=torch.long)
global_to_subset = torch.full((N_w,), -1, dtype=torch.long)
global_to_subset[subset_to_global] = torch.arange(N_subset)
subset_labels = torch.tensor(wallet_labels_np[subset_global].astype(np.int64))

active_subidx_per_step = []
for t in range(N_TIME_STEPS + 1):
    active_global_t = active_wallet[t].nonzero(as_tuple=False).squeeze(-1)
    sub_t = global_to_subset[active_global_t]
    active_subidx_per_step.append(sub_t[sub_t >= 0])

# === Move to device ===
hetero_graph = hetero_graph.to(DEVICE)
edge_active_mask = {k: v.to(DEVICE) for k, v in edge_active_mask.items()}
subset_to_global = subset_to_global.to(DEVICE)
global_to_subset = global_to_subset.to(DEVICE)
subset_labels = subset_labels.to(DEVICE)
active_subidx_per_step = [v.to(DEVICE) for v in active_subidx_per_step]
step_wallets  = [v.to(DEVICE) for v in step_wallets]
step_features = [v.to(DEVICE) for v in step_features]

# === Reporting ===
print(f"Static hetero graph  N_wallet={N_w:>9,}  N_tx={N_tx:>9,}  F_wallet={F_wallet}  F_tx={tx_features_std.shape[1]}")
for et in hetero_graph.edge_types:
    avg_active = edge_active_mask[et][1:].sum(1).float().mean().item()
    print(f"  {str(et):60s}  |E|={hetero_graph[et].edge_index.shape[1]:>9,}  avg_active/step={avg_active:>9,.0f}")
print(f"\nSubset: N_subset={N_subset:,} (illicit={len(illicit_global):,} + licit={n_licit_keep:,})")
avg_active_sub = sum(len(x) for x in active_subidx_per_step[1:]) / N_TIME_STEPS
avg_active_full = sum(len(x) for x in step_wallets[1:]) / N_TIME_STEPS
print(f"  avg active subset wallets per step: {avg_active_sub:,.0f}")
print(f"  avg active wallets per step (full graph, drives the buffer): {avg_active_full:,.0f}")

Static hetero graph  N_wallet=  822,942  N_tx=  203,769  F_wallet=55  F_tx=182
  ('wallet', 'addr_to_addr', 'wallet')                          |E|=5,553,655  avg_active/step=  116,582
  ('wallet', 'addr_to_tx', 'tx')                                |E|=  477,117  avg_active/step=    9,737
  ('tx', 'tx_to_addr', 'wallet')                                |E|=  837,124  avg_active/step=   17,084
  ('tx', 'tx_to_tx', 'tx')                                      |E|=  468,710  avg_active/step=    9,566
  ('tx', 'rev_addr_to_tx', 'wallet')                            |E|=  477,117  avg_active/step=    9,737
  ('wallet', 'rev_tx_to_addr', 'tx')                            |E|=  837,124  avg_active/step=   17,084

Subset: N_subset=54,266 (illicit=14,266 + licit=40,000)
  avg active subset wallets per step: 1,198
  avg active wallets per step (full graph, drives the buffer): 25,883


## 2. Model components

Three building blocks, then the main module that wires them together.

- **`CTLSTMCell`** — Mei & Eisner 7-gate continuous-time LSTM. Holds `(c, c_bar, delta, o)` per node; closed-form decay between events.
- **`TemporalSelfAttention`** — multi-head attention with additive time-decay bias `-softplus(beta) * (t - t_j)` and a `[B, W, H]` ring-buffer cache.
- **`HeteroGraphAttention`** — `HeteroConv` with `GATConv` on the two same-type relations and `SAGEConv` on the four cross-type relations; per-node sum across relations.

In [12]:
class CTLSTMCell(nn.Module):
    """Continuous-time LSTM (Mei & Eisner 2017). Decay is a closed-form between events;
    update is a 7-gate LSTM-like step that produces a fresh decay rate."""
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.input_to_gates = nn.Linear(input_dim, 7 * hidden_dim, bias=False)
        self.hidden_to_gates = nn.Linear(hidden_dim, 7 * hidden_dim)

    @staticmethod
    def decay(c, c_bar, delta, dt):
        # dt: [B] or [B, 1]; broadcasts over hidden dim
        if dt.ndim == 1:
            dt = dt.unsqueeze(-1)
        return c_bar + (c - c_bar) * torch.exp(-delta * dt)

    def update(self, x, c_decayed, c_bar_prev, h_decayed):
        """Apply gates at an event. Returns (c, c_bar, delta, o, h)."""
        gates = self.input_to_gates(x) + self.hidden_to_gates(h_decayed)
        i, f, z, o, i_bar, f_bar, delta = gates.chunk(7, dim=-1)
        i, f, o, i_bar, f_bar = (torch.sigmoid(g) for g in (i, f, o, i_bar, f_bar))
        z = 2 * torch.sigmoid(z) - 1                  # tanh-equivalent in [-1, 1]
        delta = F.softplus(delta)                     # non-negative decay rate
        c     = f * c_decayed + i * z
        c_bar = f_bar * c_bar_prev + i_bar * z
        h     = o * torch.tanh(c)
        return c, c_bar, delta, o, h

In [13]:
class TemporalSelfAttention(nn.Module):
    """Causal multi-head self-attention with time-decay additive bias `-softplus(beta) * dt`.
    Operates on a padded ring-buffer cache: `cache_h: [B, W, H]`, `cache_t: [B, W]`,
    `fill: [B]` (number of valid entries). `cur_t: [B]` is the query time."""
    def __init__(self, hidden_dim, n_heads=4, beta_decay_init=0.2):
        super().__init__()
        assert hidden_dim % n_heads == 0
        self.hidden_dim = hidden_dim
        self.n_heads = n_heads
        self.head_dim = hidden_dim // n_heads
        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim)
        # Per-head learnable decay rate via softplus → non-negative.
        raw_init = float(np.log(np.exp(beta_decay_init) - 1))
        self.beta_decay_raw = nn.Parameter(torch.full((n_heads,), raw_init))

    def forward(self, query, cache_h, cache_t, cur_t, fill):
        B, W, H = cache_h.shape
        if B == 0:
            return query.new_zeros(0, H)
        Q = self.q_proj(query).view(B, self.n_heads, self.head_dim)              # [B, h, d]
        K = self.k_proj(cache_h).view(B, W, self.n_heads, self.head_dim).transpose(1, 2)  # [B, h, W, d]
        V = self.v_proj(cache_h).view(B, W, self.n_heads, self.head_dim).transpose(1, 2)  # [B, h, W, d]
        scores = torch.einsum("bhd,bhwd->bhw", Q, K) / (self.head_dim ** 0.5)
        beta = F.softplus(self.beta_decay_raw)                                   # [h]
        time_diff = (cur_t.unsqueeze(-1) - cache_t).float()                      # [B, W]
        scores = scores + (-beta.view(1, -1, 1)) * time_diff.unsqueeze(1)        # [B, h, W]
        positions = torch.arange(W, device=fill.device).unsqueeze(0)             # [1, W]
        pad_mask = positions >= fill.unsqueeze(-1)                                # [B, W]; True = pad
        scores = scores.masked_fill(pad_mask.unsqueeze(1), float("-inf"))
        attn = F.softmax(scores, dim=-1)
        attn = torch.nan_to_num(attn, nan=0.0)                                    # rows with fill=0
        z = torch.einsum("bhw,bhwd->bhd", attn, V).reshape(B, H)
        return self.out_proj(z)

In [14]:
class HeteroGraphAttention(nn.Module):
    """One layer of heterogeneous attention. GATConv on same-type relations,
    SAGEConv on cross-type relations. Per-node sum across relations."""
    def __init__(self, edge_types, hidden_dim, gat_heads=2, dropout=0.2):
        super().__init__()
        same_type_keywords = ("addr_to_addr", "tx_to_tx")
        per_head = hidden_dim // gat_heads
        convs = {}
        for rel in edge_types:
            if any(kw in rel[1] for kw in same_type_keywords):
                convs[rel] = GATConv((-1, -1), per_head, heads=gat_heads, dropout=dropout, add_self_loops=False)
            else:
                convs[rel] = SAGEConv((-1, -1), hidden_dim)
        self.conv = HeteroConv(convs, aggr="sum")
        self.norms = nn.ModuleDict({
            "wallet": nn.LayerNorm(hidden_dim),
            "tx":     nn.LayerNorm(hidden_dim),
        })

    def forward(self, x_dict, edge_index_dict):
        out = self.conv(x_dict, edge_index_dict)
        return {k: F.relu(self.norms[k](v)) for k, v in out.items()}

In [15]:
class GraphAttentiveNeuralHawkes(nn.Module):
    def __init__(
        self,
        wallet_dim, tx_dim, edge_types,
        hidden_dim=64, window=10, n_attn_heads=4, gat_heads=2,
        beta_decay_init=0.2, fuse_mode="concat", dropout=0.2, n_classes=2,
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.window = window
        self.fuse_mode = fuse_mode
        self.wallet_proj = nn.Linear(wallet_dim, hidden_dim)
        self.tx_proj = nn.Linear(tx_dim, hidden_dim)
        self.graph_attn = HeteroGraphAttention(edge_types, hidden_dim, gat_heads=gat_heads, dropout=dropout)
        self.temporal_attn = TemporalSelfAttention(hidden_dim, n_heads=n_attn_heads, beta_decay_init=beta_decay_init)
        if fuse_mode == "concat":
            self.fuse = nn.Linear(2 * hidden_dim, hidden_dim)
        elif fuse_mode == "gated":
            self.fuse_gate = nn.Linear(2 * hidden_dim, hidden_dim)
            self.fuse_t = nn.Linear(hidden_dim, hidden_dim)
            self.fuse_g = nn.Linear(hidden_dim, hidden_dim)
        elif fuse_mode == "sum":
            self.fuse_t = nn.Linear(hidden_dim, hidden_dim)
            self.fuse_g = nn.Linear(hidden_dim, hidden_dim)
        else:
            raise ValueError(fuse_mode)
        self.ct_lstm = CTLSTMCell(hidden_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, n_classes)

    def fuse_attentions(self, z_time, z_graph):
        if self.fuse_mode == "concat":
            return self.fuse(torch.cat([z_time, z_graph], dim=-1))
        if self.fuse_mode == "gated":
            g = torch.sigmoid(self.fuse_gate(torch.cat([z_time, z_graph], dim=-1)))
            return g * self.fuse_t(z_time) + (1 - g) * self.fuse_g(z_graph)
        return self.fuse_t(z_time) + self.fuse_g(z_graph)

    def init_state(self, n_subset, device):
        H, W = self.hidden_dim, self.window
        return {
            "c":       torch.zeros(n_subset, H, device=device),
            "c_bar":   torch.zeros(n_subset, H, device=device),
            "delta":   torch.full((n_subset, H), 0.1, device=device),
            "o":       torch.zeros(n_subset, H, device=device),
            "last_t":  torch.zeros(n_subset, dtype=torch.long, device=device),
            "cache_h": torch.zeros(n_subset, W, H, device=device),
            "cache_t": torch.zeros(n_subset, W, dtype=torch.long, device=device),
            "fill":    torch.zeros(n_subset, dtype=torch.long, device=device),
        }

    @staticmethod
    def detach_state(state):
        return {k: (v.detach() if torch.is_tensor(v) else v) for k, v in state.items()}

    def encode_graph(self, hetero_x_wallet, hetero_x_tx, edge_index_dict_t):
        """Run one HeteroConv layer at this time step. Returns dict of [N_*, H] hidden tensors."""
        x_dict = {"wallet": self.wallet_proj(hetero_x_wallet),
                  "tx":     self.tx_proj(hetero_x_tx)}
        return self.graph_attn(x_dict, edge_index_dict_t)

    def step(self, t, z_graph_wallet_full, active_subidx, subset_to_global, state):
        """Process one time step for the active subset wallets. Returns (logits, state)."""
        if active_subidx.numel() == 0:
            return None, state
        device = z_graph_wallet_full.device
        H, W = self.hidden_dim, self.window

        # 1. Gather graph attention output for active subset wallets
        active_global = subset_to_global[active_subidx]
        z_graph = z_graph_wallet_full[active_global]                              # [B, H]

        # 2. Decay state from last_t to t, compute h_decayed
        c       = state["c"][active_subidx]
        c_bar   = state["c_bar"][active_subidx]
        delta   = state["delta"][active_subidx]
        o       = state["o"][active_subidx]
        last_t  = state["last_t"][active_subidx]
        dt = (t - last_t).clamp(min=0).float()
        c_dec = self.ct_lstm.decay(c, c_bar, delta, dt)
        h_dec = o * torch.tanh(c_dec)

        # 3. Temporal attention over past hidden states
        cache_h_act = state["cache_h"][active_subidx]
        cache_t_act = state["cache_t"][active_subidx]
        fill_act    = state["fill"][active_subidx]
        cur_t = torch.full((active_subidx.numel(),), t, dtype=torch.long, device=device)
        z_time = self.temporal_attn(h_dec, cache_h_act, cache_t_act, cur_t, fill_act)

        # 4. Fuse and update CT-LSTM
        x_fused = self.fuse_attentions(z_time, z_graph)
        c_new, c_bar_new, delta_new, o_new, h_new = self.ct_lstm.update(x_fused, c_dec, c_bar, h_dec)

        # 5. Functional state update via index_copy (differentiable; new tensors per step)
        new_state = dict(state)
        new_state["c"]      = state["c"].index_copy(0, active_subidx, c_new)
        new_state["c_bar"]  = state["c_bar"].index_copy(0, active_subidx, c_bar_new)
        new_state["delta"]  = state["delta"].index_copy(0, active_subidx, delta_new)
        new_state["o"]      = state["o"].index_copy(0, active_subidx, o_new)
        new_state["last_t"] = state["last_t"].index_copy(
            0, active_subidx, torch.full_like(active_subidx, t),
        )
        # Ring-buffer push at slot = fill % W
        slot_ring = (fill_act % W)
        flat_idx = active_subidx * W + slot_ring                                  # [B]
        cache_h_flat = state["cache_h"].view(-1, H).index_copy(0, flat_idx, h_new)
        new_state["cache_h"] = cache_h_flat.view(-1, W, H)
        cache_t_flat = state["cache_t"].view(-1).index_copy(
            0, flat_idx, torch.full_like(flat_idx, t),
        )
        new_state["cache_t"] = cache_t_flat.view(-1, W)
        new_state["fill"]    = state["fill"].index_copy(
            0, active_subidx, (fill_act + 1).clamp(max=W),
        )

        # 6. Classifier on h_new
        logits = self.classifier(h_new)
        return logits, new_state

## 3. Profiler dry-run (3 time steps)

Validates per-step wall-time before launching the full training loop. Prints which component dominates (graph attention, temporal attention, or state mgmt). Saves hours of accidentally training a too-slow model.

In [16]:
import time

model_dryrun = GraphAttentiveNeuralHawkes(
    wallet_dim=F_wallet,
    tx_dim=hetero_graph["tx"].x.shape[1],
    edge_types=hetero_graph.edge_types,
    hidden_dim=64, window=10,
).to(DEVICE)
state = model_dryrun.init_state(N_subset, DEVICE)

# Rolling causal feature buffer for wallets — at each step `t`, holds each wallet's
# most-recent feature row at time ≤ t (zeros if never seen). Updated via index_copy
# so the OLD buffer (saved by an earlier wallet_proj forward) is not mutated.
wallet_buffer = torch.zeros(N_w, F_wallet, device=DEVICE)

for t in [1, 2, 3]:
    if step_wallets[t].numel() > 0:
        wallet_buffer = wallet_buffer.index_copy(0, step_wallets[t], step_features[t])
    edge_index_dict_t = {
        rel: hetero_graph[rel].edge_index[:, edge_active_mask[rel][t]]
        for rel in hetero_graph.edge_types
    }
    t0 = time.time()
    z_graph = model_dryrun.encode_graph(wallet_buffer, hetero_graph["tx"].x, edge_index_dict_t)
    t1 = time.time()
    logits, state = model_dryrun.step(t, z_graph["wallet"], active_subidx_per_step[t], subset_to_global, state)
    t2 = time.time()
    n_active = int(active_subidx_per_step[t].numel())
    n_active_edges = sum(int(edge_active_mask[rel][t].sum()) for rel in hetero_graph.edge_types)
    print(f"t={t:2d}  active_subset_wallets={n_active:>6,}  active_edges={n_active_edges:>9,}  "
          f"graph_attn={1000*(t1-t0):>6.1f}ms  ct_lstm_step={1000*(t2-t1):>6.1f}ms")
del model_dryrun, state, wallet_buffer

t= 1  active_subset_wallets= 3,206  active_edges=  246,862  graph_attn=1598.3ms  ct_lstm_step= 101.8ms
t= 2  active_subset_wallets= 2,359  active_edges=  341,098  graph_attn=1955.5ms  ct_lstm_step=  62.1ms
t= 3  active_subset_wallets=   588  active_edges=  209,907  graph_attn=1814.6ms  ct_lstm_step=  57.1ms


## 4. Train and evaluate

- Loss: weighted CE at every wallet event (positive = illicit, weight = `n_licit_train / n_illicit_train`).
- TBPTT: detach state every K=5 steps, optimizer.step() at each TBPTT boundary.
- Train events: `t ≤ TRAIN_END`. Val events: `TRAIN_END < t ≤ VAL_END`. Test events: `t > VAL_END`.
- For each (wallet, t) event, the prediction is the CT-LSTM hidden after this event's update. We accumulate val/test predictions across all events and report illicit-class metrics.

In [17]:
TBPTT_K = 5
EPOCHS = 25
LR = 1e-3
WD = 5e-4

# Class weight from labelled events at training time steps.
train_event_pos = 0
train_event_neg = 0
for t in range(1, TRAIN_END + 1):
    sub_t = active_subidx_per_step[t]
    if sub_t.numel() == 0:
        continue
    lab_t = subset_labels[sub_t]
    train_event_pos += int((lab_t == 1).sum())
    train_event_neg += int((lab_t == 0).sum())
class_weight = torch.tensor([1.0, train_event_neg / max(train_event_pos, 1)],
                            dtype=torch.float, device=DEVICE)
print(f"Train events: pos={train_event_pos:,}  neg={train_event_neg:,}  illicit_weight={class_weight[1]:.2f}")

model = GraphAttentiveNeuralHawkes(
    wallet_dim=F_wallet,
    tx_dim=hetero_graph["tx"].x.shape[1],
    edge_types=hetero_graph.edge_types,
    hidden_dim=64, window=10, n_attn_heads=4, gat_heads=2,
    beta_decay_init=0.2, fuse_mode="concat",
).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)

def run_epoch(train_mode):
    if train_mode:
        model.train()
    else:
        model.eval()
    state = model.init_state(N_subset, DEVICE)
    # Causal rolling feature buffer for wallets. Reset every epoch.
    wallet_buffer = torch.zeros(N_w, F_wallet, device=DEVICE)
    seg_loss, seg_pos = None, 0
    epoch_loss_sum, epoch_loss_n = 0.0, 0
    val_y, val_p, test_y, test_p = [], [], [], []

    for t in range(1, N_TIME_STEPS + 1):
        # Update buffer with this step's features (causal — only past+current).
        # `index_copy` returns a new tensor, so any old buffer saved by a prior
        # `wallet_proj` forward stays intact for backward.
        if step_wallets[t].numel() > 0:
            wallet_buffer = wallet_buffer.index_copy(0, step_wallets[t], step_features[t])

        active_sub = active_subidx_per_step[t]
        if active_sub.numel() > 0:
            edge_index_dict_t = {
                rel: hetero_graph[rel].edge_index[:, edge_active_mask[rel][t]]
                for rel in hetero_graph.edge_types
            }
            ctx = torch.enable_grad() if train_mode else torch.no_grad()
            with ctx:
                z_graph = model.encode_graph(wallet_buffer, hetero_graph["tx"].x, edge_index_dict_t)
                logits, state = model.step(t, z_graph["wallet"], active_sub, subset_to_global, state)
            labels_t = subset_labels[active_sub]

            if train_mode and t <= TRAIN_END:
                step_loss = F.cross_entropy(logits, labels_t, weight=class_weight, reduction="sum")
                seg_loss = step_loss if seg_loss is None else seg_loss + step_loss
                seg_pos += int(labels_t.numel())
            elif (not train_mode) and TRAIN_END < t <= VAL_END:
                val_y.append(labels_t.detach().cpu().numpy())
                val_p.append(logits.argmax(-1).detach().cpu().numpy())
            elif (not train_mode) and t > VAL_END:
                test_y.append(labels_t.detach().cpu().numpy())
                test_p.append(logits.argmax(-1).detach().cpu().numpy())

        # TBPTT boundary: backprop through the segment, detach state + buffer.
        if train_mode and (t % TBPTT_K == 0 or t == N_TIME_STEPS):
            if seg_loss is not None and seg_pos > 0:
                optimizer.zero_grad()
                (seg_loss / seg_pos).backward()
                optimizer.step()
                epoch_loss_sum += float(seg_loss.detach()); epoch_loss_n += seg_pos
            seg_loss, seg_pos = None, 0
            state = model.detach_state(state)
            wallet_buffer = wallet_buffer.detach()

    if train_mode:
        return epoch_loss_sum / max(epoch_loss_n, 1)
    val_y_np = np.concatenate(val_y) if val_y else np.array([])
    val_p_np = np.concatenate(val_p) if val_p else np.array([])
    test_y_np = np.concatenate(test_y) if test_y else np.array([])
    test_p_np = np.concatenate(test_p) if test_p else np.array([])
    return val_y_np, val_p_np, test_y_np, test_p_np

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(train_mode=True)
    val_y, val_p, test_y, test_p = run_epoch(train_mode=False)
    val_f1  = f1_score(val_y,  val_p,  pos_label=1, zero_division=0) if len(val_y)  else 0.0
    test_f1 = f1_score(test_y, test_p, pos_label=1, zero_division=0) if len(test_y) else 0.0
    print(f"epoch {epoch:2d}  loss={train_loss:.4f}  val_illicit_F1={val_f1:.4f}  test_illicit_F1={test_f1:.4f}")

print("\nFinal VAL  classification report:")
print(classification_report(val_y,  val_p,  target_names=["licit", "illicit"], digits=4, zero_division=0))
print("Final TEST classification report:")
print(classification_report(test_y, test_p, target_names=["licit", "illicit"], digits=4, zero_division=0))

Train events: pos=7,859  neg=25,631  illicit_weight=3.26
epoch  1  loss=1.0613  val_illicit_F1=0.6784  test_illicit_F1=0.5115
epoch  2  loss=1.0074  val_illicit_F1=0.6929  test_illicit_F1=0.5287
epoch  3  loss=0.9277  val_illicit_F1=0.7063  test_illicit_F1=0.5202
epoch  4  loss=0.7907  val_illicit_F1=0.7405  test_illicit_F1=0.5647
epoch  5  loss=0.6049  val_illicit_F1=0.7571  test_illicit_F1=0.5877
epoch  6  loss=0.4406  val_illicit_F1=0.7560  test_illicit_F1=0.6341
epoch  7  loss=0.3335  val_illicit_F1=0.7900  test_illicit_F1=0.6744
epoch  8  loss=0.2921  val_illicit_F1=0.8299  test_illicit_F1=0.6856
epoch  9  loss=0.2691  val_illicit_F1=0.8361  test_illicit_F1=0.6877
epoch 10  loss=0.2528  val_illicit_F1=0.8218  test_illicit_F1=0.6870
epoch 11  loss=0.2414  val_illicit_F1=0.8076  test_illicit_F1=0.6774
epoch 12  loss=0.2220  val_illicit_F1=0.8167  test_illicit_F1=0.6415
epoch 13  loss=0.2226  val_illicit_F1=0.8271  test_illicit_F1=0.6784
epoch 14  loss=0.2102  val_illicit_F1=0.8088  

## Discussion

What each ancestor contributes:
- **Mei & Eisner CT-LSTM** — provides the continuous-time decay between wallet appearances. If the graph and temporal attention streams were dropped, what remains would be a recurrent classifier over the wallet's appearance sequence with a learned decay rate.
- **Transformer-style temporal self-attention** — lets the model pull arbitrary past states (not just the most recent) into the gate input. The additive bias `-softplus(beta) * dt` is a soft Hawkes prior: older states are exponentially down-weighted in the attention softmax.
- **Heterogeneous GAT/SAGE** — the only stream that uses graph topology. Without it, the model collapses to a per-wallet temporal sequence model with no neighbour information.

Ablation knobs exposed in `__init__`:
1. `window ∈ {5, 10, 20}` — temporal attention horizon.
2. `beta_decay_init ∈ {0.05, 0.2, 1.0}` — strong vs weak Hawkes prior.
3. `fuse_mode ∈ {concat, gated, sum}` — `gated` is the only one that lets the model *learn* when to ignore one stream.
4. `TBPTT_K` — gradient-flow horizon vs memory.

Recommended baselines to ablate against (drop into the same training loop with one stream zeroed):
- **Hawkes-only** — replace `z_graph` with zeros. If the full model doesn't beat this, the GAT branch isn't earning its complexity budget.
- **GAT-only-at-terminal** — skip the CT-LSTM entirely, run the hetero attention at each wallet's terminal time step, classify. Equivalent to the `HeteroSAGE` model in `gnn.ipynb` §6.

## Causality and leakage audit
- **Wallet GNN inputs are time-causal.** The `wallet_buffer` is a rolling causal snapshot — at step `t` it holds each wallet's most recent feature row at time `≤ t`, or zeros if the wallet hasn't appeared yet. The GNN at step `t` therefore never sees future feature rows. (An earlier draft used the wallet's *latest* time-step features as a static GNN input, which was leakage — past predictions conditioned on future feature rows. Fixed.)
- **Tx GNN inputs are time-causal by construction.** Each tx exists at exactly one time step, and `edge_active_mask` zeros out edges to txs not yet born at `t`, so a tx's features never propagate before its birth step.
- **Standardisers** are fit on rows with `Time step ≤ TRAIN_END` for both wallets (per-step rows) and txs (per-tx rows). No test-window stats leak into training.
- **CT-LSTM cell state** evolves causally from `t=1`; reset at the start of every epoch.
- **Temporal cache** only writes past `h_new` and reads past states.
- **Loss** is applied only at training-window events (`t ≤ TRAIN_END`).
- **Class weight** is computed from training-window event counts only.
- **Subset selection** uses per-wallet labels (immutable, no time data).
- **Eval forward pass** rebuilds dynamic state from `t=1` with frozen weights — no test labels enter any loss.

## Other simplifications worth flagging
- **Only subset wallets get CT-LSTM dynamics.** Unlabelled wallets contribute only through their (causal) feature buffer in the graph attention layer. Faithful to the user's design but trades off some realism for tractability.
- **Tx state is one-shot** (no CT-LSTM on tx nodes) since each tx exists at exactly one time step.